# Filtered Probes Visualization

Plots the `reproducibility_filtered_probes.json` IP list on a world map.
Coordinates are joined in from `reproducibility_probes.json` via `address_v4`.

In [1]:
# === Parameters (tagged) ===
PROBES_PATH = "datasets/ripe_atlas/filtered_probes.json"
ANCHORS_PATH = "datasets/ripe_atlas/filtered_anchors.json"

In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
print(f"repo root: {ROOT}")

# Put repo root on sys.path so `from scripts.processing.ripe_atlas...` works.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with open(ROOT / PROBES_PATH) as fh:
    raw_probes = json.load(fh)

rows = []
for e in raw_probes:
    ip = e.get("address_v4")
    geom = (e.get("geometry") or {}).get("coordinates")
    if not geom or len(geom) < 2:
        continue
    rows.append({
        "ip": ip,
        "lat": float(geom[1]),
        "lon": float(geom[0]),
        "country": e.get("country_code"),
        "asn": e.get("asn_v4"),
    })
df = pd.DataFrame(rows)

repo root: /home/nuwinslab/workspacecbg-framework


## Helper: world-map probe plotter

Factored out of the three inline blocks below. Used by the per-ASN grid at the bottom.

In [3]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature


def plot_probes_on_map(
    ax,
    lons,
    lats,
    *,
    title=None,
    label=None,
    color="#d62728",
    s=20,
    alpha=0.85,
    gridlines=True,
    background_lons=None,
    background_lats=None,
    title_fontsize=11,
):
    """Draw a PlateCarree world map on `ax` with `lons`/`lats` scered on top.

    Pass `background_lons`/`background_lats` to underplot the full corpus in
    faint gray for context — useful for per-ASN panels.
    """
    ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="#f4f1ea")
    ax.add_feature(cfeature.OCEAN, facecolor="#e8eef5")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#555")
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="#888")
    if gridlines:
        ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)
    if background_lons is not None and background_lats is not None:
        ax.scer(
            background_lons,
            background_lats,
            s=3,
            c="#bbb",
            alpha=0.25,
            edgecolors="none",
            transform=ccrs.PlateCarree(),
            zorder=2,
        )
    ax.scer(
        lons,
        lats,
        s=s,
        c=color,
        alpha=alpha,
        edgecolors="none",
        label=label,
        transform=ccrs.PlateCarree(),
        zorder=3,
    )
    if title:
        ax.set_title(title, fontsize=title_fontsize)
    if label:
        ax.legend(loc="lower left", fontsize=8, framealpha=0.85)
    return ax

## Lookups: country name + ASN operator

`country_name("US") → "United States"` and `asn_name(7922) → "AS7922 — Comcast"`.

- **Country names** come from [datasets/static_datasets/countries.json](../../../datasets/static_datasets/countries.json). That file truncates multi-word countries to the first token (e.g. `"US" → "United"`), so a fix-up dict patches the cases the corpus actually hits.
- **ASN → operator** is a hand-curated dict covering the union of top-20 probe + anchor ASNs. Unknown ASNs return just `"AS{n}"` — no online lookup.

In [4]:
with open(ROOT / "datasets/static_datasets/countries.json") as fh:
    _COUNTRIES = json.load(fh)

# countries.json names are truncated to the first whitespace token for many
# multi-word countries — patch the cases we actually see in the RIPE Atlas
# probe/anchor corpus.
_COUNTRY_FIXUPS = {
    "US": "United States", "GB": "United Kingdom", "AE": "United Arab Emirates",
    "KR": "South Korea", "KP": "North Korea", "NZ": "New Zealand",
    "HK": "Hong Kong", "MO": "Macau", "PR": "Puerto Rico",
    "DO": "Dominican Republic", "CD": "DR Congo", "CR": "Costa Rica",
    "CI": "Côte d'Ivoire", "BA": "Bosnia and Herzegovina",
    "TT": "Trinidad and Tobago", "SA": "Saudi Arabia", "LK": "Sri Lanka",
    "BF": "Burkina Faso", "PG": "Papua New Guinea",
    "GQ": "Equatorial Guinea", "GW": "Guinea-Bissau",
    "AG": "Antigua and Barbuda", "SV": "El Salvador", "CZ": "Czechia",
    "ST": "São Tomé and Príncipe", "BV": "Bouvet Island",
    "FK": "Falkland Islands", "FM": "Micronesia", "SL": "Sierra Leone",
    "NC": "New Caledonia", "VG": "British Virgin Islands",
    "VI": "U.S. Virgin Islands", "MP": "Northern Mariana Islands",
    "CV": "Cape Verde", "SH": "Saint Helena", "CK": "Cook Islands",
    "MH": "Marshall Islands", "SB": "Solomon Islands",
    "HM": "Heard and McDonald Islands", "TF": "French Southern Territories",
    "TC": "Turks and Caicos", "CC": "Cocos (Keeling) Islands",
    "CX": "Christmas Island", "EH": "Western Sahara",
    "WF": "Wallis and Futuna", "FO": "Faroe Islands", "IM": "Isle of Man",
    "PF": "French Polynesia", "GF": "French Guiana",
    "PM": "Saint Pierre and Miquelon", "SM": "San Marino",
    "KN": "Saint Kitts and Nevis", "LC": "Saint Lucia",
    "VC": "Saint Vincent and the Grenadines",
    "GS": "South Georgia", "AS": "American Samoa",
    "ZA": "South Africa", "CF": "Central African Republic",
}


def country_name(code):
    """ISO 3166-1 alpha-2 → full country name; falls back to the code itself."""
    if not code:
        return ""
    if code in _COUNTRY_FIXUPS:
        return _COUNTRY_FIXUPS[code]
    entry = _COUNTRIES.get(code)
    if not entry:
        return code
    return entry.get("name") or code


# Hand-curated AS number → operator. Covers the union of top-20 ASNs seen in
# the probe and anchor corpora. Unknown ASNs return only "AS{n}".
_ASN_OPERATORS = {
    # National-telco / eyeball
    7922:  "Comcast",
    3320:  "Deutsche Telekom",
    12322: "Free SAS",
    3209:  "Vodafone DE",
    3215:  "Orange FR",
    2860:  "NOS Comunicações",
    7018:  "",
    701:   "Verizon",
    1136:  "KPN",
    15557: "SFR",
    33915: "Vodafone NL (Ziggo)",
    5089:  "Virgin Media UK",
    2856:  "BT",
    4764:  "TPG Telecom",
    5410:  "Bouygues Telecom",
    3352:  "Telefónica España",
    6830:  "Liberty Global",
    8881:  "1&1 Versatel",
    48503: "Kazakhtelecom",
    680:   "DFN (German Research Network)",
    39138: "Open Carrier (DE)",
    # Cloud / CDN / hyperscaler
    16509: " AWS",
    47583: "Hostinger",
    20473: "Vultr (Choopa)",
    31898: "Oracle Cloud",
    396982: " Cloud",
    12008: "Neustar / Edgio",
    202422: "G-Core Labs",
    202196: "G-Core Labs",
    31713: "Edgio (Edgecast)",
    14061: "DigitalOcean",
    16276: "OVH",
    3491:  "PCCW Global",
    36236: "NetActuate",
    42473: "ANEXIA",
    137409: "GSL Networks (AU)",
    208722: "Mevspace",
    15133: "Edgio (Verizon Media)",
}


def asn_operator(asn):
    """AS number → operator name, or empty string if not in the curated dict."""
    try:
        n = int(asn)
    except (TypeError, ValueError):
        return ""
    return _ASN_OPERATORS.get(n, "")


def asn_name(asn):
    """AS number → 'AS{n} — Operator' (or just 'AS{n}' if unknown)."""
    try:
        n = int(asn)
    except (TypeError, ValueError):
        return str(asn)
    op = _ASN_OPERATORS.get(n)
    return f"AS{n} — {op}" if op else f"AS{n}"


# quick demo
print("country_name('US')    →", country_name("US"))
print("country_name('GB')    →", country_name("GB"))
print("country_name('XX')    →", country_name("XX"))   # unknown
print("asn_name(7922)        →", asn_name(7922))
print("asn_name(396982)      →", asn_name(396982))
print("asn_name(999999)      →", asn_name(999999))     # unknown
print("asn_operator(20473)   →", repr(asn_operator(20473)))

country_name('US')    → United States
country_name('GB')    → United Kingdom
country_name('XX')    → XX
asn_name(7922)        → AS7922 — Comcast
asn_name(396982)      → AS396982 —  Cloud
asn_name(999999)      → AS999999
asn_operator(20473)   → 'Vultr (Choopa)'


## World map: all probes

In [5]:
all_rows = []
for e in raw_probes:
    geom = (e.get("geometry") or {}).get("coordinates")
    if not geom or len(geom) < 2:
        continue
    all_rows.append({
        "ip": e.get("address_v4"),
        "lat": float(geom[1]),
        "lon": float(geom[0]),
        "country": e.get("country_code"),
        "asn": e.get("asn_v4"),
    })
all_df = pd.DataFrame(all_rows)
print(f"all probes with coords: {len(all_df)}")

fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#f4f1ea")
ax.add_feature(cfeature.OCEAN, facecolor="#e8eef5")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#555")
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="#888")
ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)

ax.scer(all_df["lon"], all_df["lat"], s=22, c="#d62728", alpha=0.9,
           edgecolors="none", label=f"filtered probe  (n={len(all_df)})",
           transform=ccrs.PlateCarree(), zorder=3)
ax.set_title("reproducibility_probes — all probes")
ax.legend(loc="lower left", fontsize=9, framealpha=0.85)
plt.tight_layout()
plt.show()

all probes with coords: 9229


## World map: reproducibility anchors

In [6]:
with open(ROOT / ANCHORS_PATH) as fh:
    raw_anchors = json.load(fh)

anchor_rows = []
for e in raw_anchors:
    geom = (e.get("geometry") or {}).get("coordinates")
    if not geom or len(geom) < 2:
        continue
    anchor_rows.append({
        "ip": e.get("address_v4"),
        "lat": float(geom[1]),
        "lon": float(geom[0]),
        "country": e.get("country_code"),
        "asn": e.get("asn_v4"),
    })
anchors_df = pd.DataFrame(anchor_rows)
print(f"anchors with coords: {len(anchors_df)}")

fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#f4f1ea")
ax.add_feature(cfeature.OCEAN, facecolor="#e8eef5")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#555")
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="#888")
ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)
ax.scer(anchors_df["lon"], anchors_df["lat"], s=18, c="#2ca02c", alpha=0.85,
           edgecolors="none", label=f"anchor  (n={len(anchors_df)})",
           transform=ccrs.PlateCarree(), zorder=3)
ax.set_title("reproducibility_anchors — geographic distribution")
ax.legend(loc="lower left", fontsize=9, framealpha=0.85)
plt.tight_layout()
plt.show()

anchors with coords: 723


## Top-20 probe ASNs

Probes-by-ASN concentration on the RIPE Atlas corpus. A long tail: top-20 ASNs
typically cover only ~25% of the fleet, with most ASNs hosting ≤10 probes.

In [7]:
# Per-ASN aggregate on the full probe corpus (all_df, built above).
total_vps = len(all_df)
n_unique_asns = all_df["asn"].nunique()

by_asn = (
    all_df.groupby("asn")
    .agg(n_probes=("ip", "size"), n_countries=("country", "nunique"))
    .sort_values("n_probes", ascending=False)
)
by_asn["pct"] = (by_asn["n_probes"] / total_vps * 100).round(2)

# Top-3 probe-host countries per ASN as a single "Country(n), ..." string.
# Anchor-side counterpart is added in the next cell as `anchor_top_countries`.
country_counts = (
    all_df.groupby(["asn", "country"]).size().reset_index(name="n")
    .sort_values(["asn", "n"], ascending=[True, False])
)
probe_top_countries = (
    country_counts.groupby("asn")
    .apply(
        lambda d: ", ".join(f"{country_name(r.country)}({r.n})" for r in d.head(3).itertuples()),
        include_groups=False,
    )
    .rename("probe_top_countries")
)

top20 = by_asn.head(20).join(probe_top_countries)
top20.insert(0, "operator", top20.index.map(asn_operator))

print(f"Total VPs: {total_vps:,}")
print(f"Unique ASNs: {n_unique_asns:,}")
print(f"Top-20 ASN share of corpus: {top20['pct'].sum():.2f}%")
print(f"  top-1:  {top20.iloc[0]['pct']:.2f}%")
print(f"  top-5:  {top20.head(5)['pct'].sum():.2f}%")
print(f"  top-10: {top20.head(10)['pct'].sum():.2f}%")
print()
top20

Total VPs: 9,229
Unique ASNs: 2,990
Top-20 ASN share of corpus: 26.08%
  top-1:  2.99%
  top-5:  12.14%
  top-10: 19.73%



,operator,n_probes,n_countries,pct,probe_top_countries
asn,,,,,
7922,Comcast,276,2,2.99,"United States(275), United Kingdom(1)"
3320,Deutsche Telekom,219,1,2.37,Germany(219)
12322,Free SAS,217,1,2.35,France(217)
3215,Orange FR,206,2,2.23,"France(196), Réunion(10)"
3209,Vodafone DE,203,2,2.20,"Germany(202), Switzerland(1)"
2860,NOS Comunicações,176,1,1.91,Portugal(176)
7018,AT&T,154,1,1.67,United States(154)
47583,Hostinger,130,9,1.41,"United States(27), France(15), India(15)"
701,Verizon,124,2,1.34,"United States(123), Canada(1)"


## Unique accessible anchors per ASN  (ClickHouse)

Joins the top-20 probe ASNs against `ping_10k_to_anchors` to count, for
each ASN, how many **distinct anchor IPs** are reachable by *any* of that
ASN's probes with a valid ping (`min > 0`, `min < 10000` ms). This is the
ceiling on per-ASN K-fold eval coverage: an ASN that reaches only 25% of
the anchor corpus will have a merged-folds `succ/total` denominator of
~0.25 × 721 anchors regardless of fold layout. Requires ClickHouse
reachable via `.env`.

In [8]:
import ipaddress
from scripts.utils.clickhouse import Clickhouse

# RTT upper bound (ms). Matches `_DEFAULT_THRESHOLD = 70` in
# scripts/benchmark/v2/sources/ripe_atlas_asn_corpora.py — the same bound the
# per-fold eval applies, so this count is the actual ceiling on per-ASN
# K-fold eval coverage. Raise to e.g. 10000 to count any successful ping.
RTT_THRESHOLD_MS = 10000

# Restrict the CH query to probes in the top-20 ASNs.
top20_asn_set = set(top20.index.tolist())
top20_probe_ips = (
    all_df[all_df["asn"].isin(top20_asn_set)]["ip"].dropna().unique()
)
src_nums = tuple(int(ipaddress.IPv4Address(ip)) for ip in top20_probe_ips)

ch = Clickhouse()
query = f"""
    SELECT
        IPv4NumToString(src) AS src_ip,
        IPv4NumToString(dst) AS dst_ip
    FROM {ch.database}.ping_10k_to_anchors
    WHERE `min` > -1 AND `min` < %(rtt_max)s
      AND dst != src
      AND src IN %(srcs)s
    GROUP BY src, dst
"""
rows = list(ch.client.execute_iter(
    query, params={"srcs": src_nums, "rtt_max": RTT_THRESHOLD_MS},
))
ch.client.disconnect()

probe_anchor_pairs = pd.DataFrame(rows, columns=["src_ip", "dst_ip"])
print(
    f"Pulled {len(probe_anchor_pairs):,} distinct (probe, anchor) pairs at "
    f"`min` < {RTT_THRESHOLD_MS} ms across "
    f"{probe_anchor_pairs['src_ip'].nunique()} probes and "
    f"{probe_anchor_pairs['dst_ip'].nunique()} anchors"
)

# Map probe_ip → ASN, then per-ASN union of reachable anchors.
# Dedupe by IP — the probe JSON occasionally lists the same IP twice with the
# same ASN; a non-unique index breaks .map().
probe_to_asn = all_df.drop_duplicates(subset="ip").set_index("ip")["asn"]
probe_anchor_pairs["asn"] = probe_anchor_pairs["src_ip"].map(probe_to_asn)
unique_anchors_per_asn = (
    probe_anchor_pairs.groupby("asn")["dst_ip"].nunique()
    .rename("unique_anchor_count")
)

# Top-3 anchor-host countries reachable per ASN — the eval-side analogue of
# `probe_top_countries`. Counts distinct anchor IPs per country, picks top-3.
anchor_country = (
    anchors_df.drop_duplicates(subset="ip").set_index("ip")["country"]
)
probe_anchor_pairs["dst_country"] = probe_anchor_pairs["dst_ip"].map(anchor_country)
anchor_country_counts = (
    probe_anchor_pairs.dropna(subset=["dst_country"])
    .drop_duplicates(subset=["asn", "dst_ip"])  # one row per (asn, anchor)
    .groupby(["asn", "dst_country"]).size().reset_index(name="n")
    .sort_values(["asn", "n"], ascending=[True, False])
)
anchor_top_countries = (
    anchor_country_counts.groupby("asn")
    .apply(
        lambda d: ", ".join(f"{country_name(r.dst_country)}({r.n})" for r in d.head(3).itertuples()),
        include_groups=False,
    )
    .rename("anchor_top_countries")
)

# ach to top20; fill 0/"" for any ASN with no reachable anchors.
top20 = (
    top20.drop(
        columns=["unique_anchor_count", "anchor_top_countries"],
        errors="ignore",
    )
    .join(unique_anchors_per_asn)
    .join(anchor_top_countries)
    .fillna({"unique_anchor_count": 0, "anchor_top_countries": ""})
    .astype({"unique_anchor_count": int})
)

display_cols = [
    "operator", "n_probes", "unique_anchor_count",
    "n_countries", "pct",
    "probe_top_countries", "anchor_top_countries",
]
top20[display_cols].sort_values(
    ["unique_anchor_count", "n_probes"], ascending=[False, False],
)

Pulled 1,766,598 distinct (probe, anchor) pairs at `min` < 10000 ms across 2336 probes and 783 anchors


,operator,n_probes,unique_anchor_count,n_countries,pct,probe_top_countries,anchor_top_countries
asn,,,,,,,
701,Verizon,124,781,2,1.34,"United States(123), Canada(1)","United States(98), Germany(97), Netherlands(43)"
3320,Deutsche Telekom,219,780,1,2.37,Germany(219),"United States(98), Germany(97), Netherlands(43)"
12322,Free SAS,217,780,1,2.35,France(217),"United States(98), Germany(97), Netherlands(43)"
2860,NOS Comunicações,176,780,1,1.91,Portugal(176),"United States(98), Germany(97), Netherlands(43)"
6830,Liberty Global,54,780,6,0.59,"Ireland(30), Switzerland(17), Poland(4)","United States(98), Germany(97), Netherlands(43)"
7922,Comcast,276,779,2,2.99,"United States(275), United Kingdom(1)","United States(98), Germany(97), Netherlands(43)"
3215,Orange FR,206,779,2,2.23,"France(196), Réunion(10)","United States(98), Germany(97), Netherlands(43)"
3209,Vodafone DE,203,779,2,2.20,"Germany(202), Switzerland(1)","United States(98), Germany(97), Netherlands(43)"
7018,AT&T,154,779,1,1.67,United States(154),"United States(98), Germany(97), Netherlands(43)"


## Top-20 ASN probes — per-ASN world maps

One panel per ASN, ordered by probe count. Faint gray dots underneath show the
full corpus for spatial context. Single-country national-telco ASNs look like
dense national clusters; cloud/hyperscaler ASNs (AWS, Vultr, Oracle) look like
sparse globally-scered datacenter footprints.

In [9]:
# Reachable-anchor (green) + probe (red) overlay for each top-20 ASN.
# Anchors are looked up from `probe_anchor_pairs` (built in the previous cell
# at `min` < RTT_THRESHOLD_MS) → anchors_df for coords. Both layers carry
# alpha so overlapping dots stay visible.
top20_asns = top20.index.tolist()

# Anchor-coord lookup keyed by IP (dedup by IP to keep a unique index).
anchor_coords = (
    anchors_df.drop_duplicates(subset="ip").set_index("ip")[["lat", "lon"]]
)

n_cols = 4
n_rows = 5
fig = plt.figure(figsize=(22, 18))

for i, asn in enumerate(top20_asns):
    probes = all_df[all_df["asn"] == asn]
    n_probes = len(probes)
    n_probe_countries = probes["country"].nunique()
    probe_word = "country" if n_probe_countries == 1 else "countries"

    # Reachable anchors: distinct dst IPs paired with this ASN's probes.
    reachable_ips = (
        probe_anchor_pairs.loc[probe_anchor_pairs["asn"] == asn, "dst_ip"]
        .unique()
    )
    anchors_sub = anchor_coords.loc[
        anchor_coords.index.intersection(reachable_ips)
    ]
    n_anchors = len(anchors_sub)

    ax = fig.add_subplot(n_rows, n_cols, i + 1, projection=ccrs.PlateCarree())
    ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="#f4f1ea")
    ax.add_feature(cfeature.OCEAN, facecolor="#e8eef5")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#555")
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="#888")

    # Anchors first (green, semi-transparent).
    ax.scer(
        anchors_sub["lon"].to_numpy(),
        anchors_sub["lat"].to_numpy(),
        s=14, c="#2ca02c", alpha=0.55, edgecolors="none",
        transform=ccrs.PlateCarree(), zorder=2,
        label=f"anchor reachable (n={n_anchors})",
    )
    # Probes on top (red, also semi-transparent so dense clusters stay legible).
    ax.scer(
        probes["lon"].to_numpy(),
        probes["lat"].to_numpy(),
        s=14, c="#d62728", alpha=0.75, edgecolors="none",
        transform=ccrs.PlateCarree(), zorder=3,
        label=f"probe (n={n_probes})",
    )
    ax.set_title(
        f"{asn_name(asn)}  probes={n_probes}, anchors={n_anchors}  "
        f"({n_probe_countries} {probe_word})",
        fontsize=10,
    )
    ax.legend(loc="lower left", fontsize=7, framealpha=0.85)

fig.suptitle(
    f"Top-20 RIPE Atlas probe ASNs — reachable anchors (green) + probes (red); "
    f"reach threshold `min` < {RTT_THRESHOLD_MS} ms",
    fontsize=14, y=0.995,
)
plt.tight_layout()
plt.show()

## Top-20 probe ASNs reranked by **city diversity**

Within each of the top-20 probe ASNs (ranked by raw probe count above),
deduplicate probes that sit in the same **city**, then rerank the same 20
ASNs by unique-city count. "City" is defined by binning lat/lon to a 0.1°
grid (~11 km at the equator) — coarse enough that a metro collapses to one
cell, fine enough to separate nearby cities.

Why this mers for VP-corpus design: an ASN with 337 probes packed into
40 US metros is operationally a 40-VP fleet — co-located probes contribute
redundant RTT samples, not independent measurements. Reranking by city
count exposes which ASNs offer genuine geographic *coverage* vs which just
have a lot of probes in a few places.

In [10]:
# City definition: lat/lon snapped to a 0.1° grid (~11 km at the equator).
CITY_GRID_DEG = 0.1

probes_with_city = all_df.copy()
probes_with_city["city_lat"] = (probes_with_city["lat"] / CITY_GRID_DEG).round() * CITY_GRID_DEG
probes_with_city["city_lon"] = (probes_with_city["lon"] / CITY_GRID_DEG).round() * CITY_GRID_DEG

# Restrict to the top-20 ASNs already computed above.
top20_asn_set = set(top20.index.tolist())
top20_probes_city = probes_with_city[probes_with_city["asn"].isin(top20_asn_set)]

# City-deduped probe set per ASN — same selection rule as
# scripts/processing/ripe_atlas/select_probes_and_anchors.py applies before
# materializing the per-ASN benchmark corpora. Keep one probe per
# (asn, city_lat, city_lon) bin so n_cities is also the number of kept probes.
deduped_probes_per_asn = (
    top20_probes_city.drop_duplicates(subset=["asn", "city_lat", "city_lon"])
)
city_stats = deduped_probes_per_asn.groupby("asn").agg(n_cities=("ip", "size"))

# Re-count `unique_anchor_count` against the city-deduped probe set: the
# upstream cell's value uses *all* probes of the ASN, but the benchmark VP
# selection drops same-city duplicates. Re-joining `probe_anchor_pairs`
# restricted to the kept probes gives the anchor reach the benchmark
# actually sees. (At RTT_THRESHOLD_MS=10000 the delta is ≤2 anchors per
# ASN — co-located probes have near-identical reach when any successful
# ping counts — but at tighter thresholds the divergence can grow.)
deduped_probe_ips = set(deduped_probes_per_asn["ip"])
unique_anchor_count_dedup = (
    probe_anchor_pairs[probe_anchor_pairs["src_ip"].isin(deduped_probe_ips)]
    .groupby("asn")["dst_ip"].nunique()
    .rename("unique_anchor_count")
)

# Reuse the existing top20 columns (operator, n_probes, n_countries, pct,
# probe_top_countries, anchor_top_countries) but DROP the all-probes
# `unique_anchor_count` so the city-deduped value takes its place.
top20_by_cities = (
    top20.drop(columns=["unique_anchor_count"], errors="ignore")
    .join(city_stats)
    .join(unique_anchor_count_dedup)
)
top20_by_cities["probes_per_city"] = (
    top20_by_cities["n_probes"] / top20_by_cities["n_cities"]
).round(1)

# Rerank by (n_cities, unique_anchor_count) DESC — city count captures
# *geographic* coverage; unique_anchor_count is the data-availability ceiling
# on eval coverage. Together they pick ASNs that are both well-spread and
# actively measuring the anchor corpus.
top20_by_cities = top20_by_cities.sort_values(
    ["n_cities", "unique_anchor_count"], ascending=[False, False],
)

display_cols = [
    "operator", "n_probes", "n_cities", "probes_per_city",
    "unique_anchor_count", "n_countries", "pct",
    "probe_top_countries", "anchor_top_countries",
]
top20_by_cities = top20_by_cities[display_cols]

print(f"City grid: {CITY_GRID_DEG}° (~{int(CITY_GRID_DEG*111)} km)")
print(f"Top-20 ASN unique cities — total: {top20_by_cities['n_cities'].sum():,}")
print(f"  raw probes in same 20 ASNs:    {top20_by_cities['n_probes'].sum():,}")
print(f"  city-dedup compression ratio:  "
      f"{top20_by_cities['n_probes'].sum() / top20_by_cities['n_cities'].sum():.1f}x")
print()
top20_by_cities

City grid: 0.1° (~11 km)
Top-20 ASN unique cities — total: 1,604
  raw probes in same 20 ASNs:    2,407
  city-dedup compression ratio:  1.5x



,operator,n_probes,n_cities,probes_per_city,unique_anchor_count,n_countries,pct,probe_top_countries,anchor_top_countries
asn,,,,,,,,,
7922,Comcast,276,219,1.3,779,2,2.99,"United States(275), United Kingdom(1)","United States(98), Germany(97), Netherlands(43)"
3320,Deutsche Telekom,219,170,1.3,780,1,2.37,Germany(219),"United States(98), Germany(97), Netherlands(43)"
3209,Vodafone DE,203,164,1.2,779,2,2.20,"Germany(202), Switzerland(1)","United States(98), Germany(97), Netherlands(43)"
3215,Orange FR,206,151,1.4,779,2,2.23,"France(196), Réunion(10)","United States(98), Germany(97), Netherlands(43)"
12322,Free SAS,217,146,1.5,779,1,2.35,France(217),"United States(98), Germany(97), Netherlands(43)"
7018,AT&T,154,125,1.2,779,1,1.67,United States(154),"United States(98), Germany(97), Netherlands(43)"
701,Verizon,124,91,1.4,780,2,1.34,"United States(123), Canada(1)","United States(98), Germany(97), Netherlands(43)"
1136,KPN,116,82,1.4,779,2,1.26,"Netherlands(115), United States(1)","United States(98), Germany(97), Netherlands(43)"
15557,SFR,71,64,1.1,779,1,0.77,France(71),"United States(98), Germany(97), Netherlands(43)"


## Top-20 probe ASNs (city-deduplicated) — per-ASN world maps

Panels ordered by **unique-city count** (the reranking above). Each red dot
is one city — at most one per 0.1° grid bin per ASN. Faint gray dots show
the full corpus's city centers for context. ASNs that look spatially
"empty" had a high `probes_per_city` ratio above — i.e. lots of probes
clustered in a few places.

In [11]:
top20_by_cities_asns = top20_by_cities.index.tolist()

# Corpus background: one dot per city across ALL probes (not just top-20).
corpus_cities = probes_with_city.drop_duplicates(subset=["city_lat", "city_lon"])
corpus_bg_lons = corpus_cities["city_lon"].to_numpy()
corpus_bg_lats = corpus_cities["city_lat"].to_numpy()

n_cols = 4
n_rows = 5
fig = plt.figure(figsize=(22, 18))

for i, asn in enumerate(top20_by_cities_asns):
    sub_probes = top20_probes_city[top20_probes_city["asn"] == asn]
    sub_cities = sub_probes.drop_duplicates(subset=["city_lat", "city_lon"])
    n_cities = len(sub_cities)
    n_probes = len(sub_probes)
    n_countries = sub_cities["country"].nunique()
    country_word = "country" if n_countries == 1 else "countries"
    ax = fig.add_subplot(n_rows, n_cols, i + 1, projection=ccrs.PlateCarree())
    plot_probes_on_map(
        ax,
        sub_cities["city_lon"].to_numpy(),
        sub_cities["city_lat"].to_numpy(),
        title=(
            f"{asn_name(asn)}  cities={n_cities}  "
            f"(probes={n_probes}, {n_countries} {country_word})"
        ),
        color="#d62728",
        s=18,
        alpha=0.9,
        gridlines=False,
        background_lons=corpus_bg_lons,
        background_lats=corpus_bg_lats,
        title_fontsize=10,
    )

fig.suptitle(
    f"Top-20 probe ASNs reranked by city diversity  "
    f"(0.1° grid; corpus: {len(corpus_cities):,} unique cities)",
    fontsize=14,
    y=0.995,
)
plt.tight_layout()
plt.show()

## Top-20 anchor ASNs

Same view for the anchor corpus. Anchors are far fewer than probes (~700 vs
~12K), so per-ASN counts are single- to low-double-digit. The concentration
mers more for eval — these are the targets the K-fold protocol holds out,
and ASN-blocked stratification would key off this distribution.

In [12]:
# Per-ASN aggregate on the anchor corpus (anchors_df, built above).
total_anchors = len(anchors_df)
n_unique_anchor_asns = anchors_df["asn"].nunique()

anchor_by_asn = (
    anchors_df.groupby("asn")
    .agg(n_anchors=("ip", "size"), n_countries=("country", "nunique"))
    .sort_values("n_anchors", ascending=False)
)
anchor_by_asn["pct"] = (anchor_by_asn["n_anchors"] / total_anchors * 100).round(2)

anchor_country_counts = (
    anchors_df.groupby(["asn", "country"]).size().reset_index(name="n")
    .sort_values(["asn", "n"], ascending=[True, False])
)
anchor_top_countries = (
    anchor_country_counts.groupby("asn")
    .apply(
        lambda d: ", ".join(f"{country_name(r.country)}({r.n})" for r in d.head(3).itertuples()),
        include_groups=False,
    )
    .rename("top_countries")
)

anchor_top20 = anchor_by_asn.head(20).join(anchor_top_countries)
anchor_top20.insert(0, "operator", anchor_top20.index.map(asn_operator))

print(f"Total anchors: {total_anchors:,}")
print(f"Unique anchor ASNs: {n_unique_anchor_asns:,}")
print(f"Top-20 ASN share of anchor corpus: {anchor_top20['pct'].sum():.2f}%")
print(f"  top-1:  {anchor_top20.iloc[0]['pct']:.2f}%")
print(f"  top-5:  {anchor_top20.head(5)['pct'].sum():.2f}%")
print(f"  top-10: {anchor_top20.head(10)['pct'].sum():.2f}%")
print()
anchor_top20

Total anchors: 723
Unique anchor ASNs: 518
Top-20 ASN share of anchor corpus: 22.80%
  top-1:  2.49%
  top-5:  10.64%
  top-10: 15.90%



,operator,n_anchors,n_countries,pct,top_countries
asn,,,,,
396982,Cloud,18,14,2.49,"United States(4), Japan(2), Australia(1)"
20473,Vultr (Choopa),17,11,2.35,"United States(7), Australia(1), Canada(1)"
12008,Neustar / Edgio,16,15,2.21,"India(2), United Arab Emirates(1), Australia(1)"
202422,G-Core Labs,15,14,2.07,"United States(2), Australia(1), Brazil(1)"
48503,Kazakhtelecom,11,1,1.52,Kazakhstan(11)
31713,Edgio (Edgecast),10,9,1.38,"United States(2), United Arab Emirates(1), Uni..."
208722,Mevspace,7,6,0.97,"Russia(2), Germany(1), Finland(1)"
42473,ANEXIA,7,7,0.97,"United Arab Emirates(1), Austria(1), Chile(1)"
36236,NetActuate,7,5,0.97,"United States(3), Brazil(1), Chile(1)"


## Top-20 anchor ASNs — per-ASN world maps

One panel per anchor ASN, ordered by anchor count. Green dots = anchors of
that ASN; faint gray = full anchor corpus for spatial context. Cloud/CDN ASNs
(AWS, Vultr, OVH) dominate the anchor corpus and tend to span many countries.

In [13]:
anchor_top20_asns = anchor_top20.index.tolist()

n_cols = 4
n_rows = 5
fig = plt.figure(figsize=(22, 18))

anchor_bg_lons = anchors_df["lon"].to_numpy()
anchor_bg_lats = anchors_df["lat"].to_numpy()

for i, asn in enumerate(anchor_top20_asns):
    sub = anchors_df[anchors_df["asn"] == asn]
    n = len(sub)
    n_countries = sub["country"].nunique()
    country_word = "country" if n_countries == 1 else "countries"
    ax = fig.add_subplot(n_rows, n_cols, i + 1, projection=ccrs.PlateCarree())
    plot_probes_on_map(
        ax,
        sub["lon"].to_numpy(),
        sub["lat"].to_numpy(),
        title=f"{asn_name(asn)}  n={n}  ({n_countries} {country_word})",
        color="#2ca02c",
        s=18,
        alpha=0.9,
        gridlines=False,
        background_lons=anchor_bg_lons,
        background_lats=anchor_bg_lats,
        title_fontsize=10,
    )

fig.suptitle(
    f"Top-20 RIPE Atlas anchor ASNs  (corpus: {total_anchors:,} anchors across {n_unique_anchor_asns:,} ASNs)",
    fontsize=14,
    y=0.995,
)
plt.tight_layout()
plt.show()

## Anchors by continent

Coarsest geographic view of the eval corpus: how many anchors sit in each
continent. Useful as a sanity check before ASN- or spatially-blocked
stratification — folds drawn from a continent with few anchors will be
small, and any "global" deployment-scenario claims have to account for
where the bulk of the corpus actually lives.

In [14]:
# Continent mapping comes from scripts/processing/ripe_atlas/continents.py
# (shared with the select_probes_and_anchors.py CLI).
from scripts.processing.ripe_atlas.continents import continent_of

anchors_by_continent = (
    anchors_df.assign(continent=anchors_df["country"].map(continent_of))
    .groupby("continent")
    .agg(n_anchors=("ip", "size"), n_countries=("country", "nunique"))
    .sort_values("n_anchors", ascending=False)
)
anchors_by_continent["pct"] = (
    anchors_by_continent["n_anchors"] / len(anchors_df) * 100
).round(2)

# Top-3 countries per continent for context
cc_rows = (
    anchors_df.assign(continent=anchors_df["country"].map(continent_of))
    .groupby(["continent", "country"]).size().reset_index(name="n")
    .sort_values(["continent", "n"], ascending=[True, False])
)
top3_per_cont = (
    cc_rows.groupby("continent")
    .apply(
        lambda d: ", ".join(f"{country_name(r.country)}({r.n})" for r in d.head(3).itertuples()),
        include_groups=False,
    )
    .rename("top_countries")
)
anchors_by_continent = anchors_by_continent.join(top3_per_cont)

unknown = anchors_df[anchors_df["country"].map(continent_of) == "Unknown"]["country"].unique()
if len(unknown):
    print(f"WARNING: unknown country codes hit continent fallback: {list(unknown)}")

print(f"Total anchors:    {len(anchors_df):,}")
print(f"Total continents: {anchors_by_continent.index.nunique()} "
      f"(of 7 standard + 'Unknown')")
print()
anchors_by_continent

Total anchors:    723
Total continents: 6 (of 7 standard + 'Unknown')



,n_anchors,n_countries,pct,top_countries
continent,,,,
Europe,420,36,58.09,"Germany(99), Netherlands(43), France(39)"
North America,127,8,17.57,"United States(101), Canada(17), Mexico(3)"
Asia,115,29,15.91,"Singapore(22), Kazakhstan(13), United Arab Emi..."
South America,27,9,3.73,"Brazil(10), Chile(5), Argentina(4)"
Oceania,18,3,2.49,"Australia(13), New Zealand(4), New Caledonia(1)"
Africa,16,9,2.21,"South Africa(6), Ghana(2), Mauritius(2)"


In [15]:
fig, ax = plt.subplots(figsize=(10, 4.5))
order = anchors_by_continent.sort_values("n_anchors", ascending=True)
bars = ax.barh(order.index, order["n_anchors"], color="#2ca02c", alpha=0.85)
for bar, n, pct in zip(bars, order["n_anchors"], order["pct"]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"{n}  ({pct:.1f}%)", va="center", fontsize=9)
ax.set_xlabel("anchors", fontsize=11)
ax.set_xlim(0, order["n_anchors"].max() * 1.15)
ax.set_title(
    f"RIPE Atlas anchors by continent  "
    f"(n={len(anchors_df)}, {anchors_df['country'].nunique()} countries)",
    fontsize=12,
)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", linewidth=0.3, alpha=0.4)
plt.tight_layout()
plt.show()

## Benchmark inputs: shared anchor corpus

Loads the **shared** anchor eval set produced by
[scripts/processing/ripe_atlas/select_probes_and_anchors.py](../scripts/processing/ripe_atlas/select_probes_and_anchors.py).
The script excludes the *union* of all six setup ASNs from the 723-anchor
sanitized corpus, so every setup evaluates against the same 721 anchors —
s-to-s comparison, no institutional-proximity leakage in any
direction. The map below marks the dropped anchors (×) so the methodology
guard is visually traceable.

In [16]:
CORPORA_DIR = ROOT / "datasets/ripe_atlas/asn_corpora"

with open(CORPORA_DIR / "anchors.json") as fh:
    shared_anchors = json.load(fh)
with open(CORPORA_DIR / "anchors_stats.json") as fh:
    shared_anchor_stats = json.load(fh)

print(f"Shared eval anchors: {shared_anchor_stats['kept']:,} "
      f"(from {shared_anchor_stats['input_total']:,} sanitized "
      f"minus {shared_anchor_stats['dropped_count']:,} same-ASN)")
print(f"Excluded ASNs: {shared_anchor_stats['excluded_asns']}")
print()
print("Kept anchors by continent:")
for cont, n in shared_anchor_stats["kept_by_continent"].items():
    print(f"  {cont:<15s} {n}")
print()
print(f"Dropped anchors ({len(shared_anchor_stats['dropped_anchors'])}):")
for d in shared_anchor_stats["dropped_anchors"]:
    operator = shared_anchor_stats["excluded_asn_operators"].get(str(d["asn_v4"]), "?")
    print(f"  id={d['id']:>5}  {d['address_v4']:>16}  AS{d['asn_v4']} ({operator})  cc={d['country_code']}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/nuwinslab/workspacecbg-framework/datasets/ripe_atlas/asn_corpora/anchors.json'

In [17]:
# Plot the 721 shared anchors + mark the 2 dropped ones with red ×.
def _coords(entries):
    lons, lats = [], []
    for e in entries:
        geom = (e.get("geometry") or {}).get("coordinates")
        if not geom or len(geom) < 2:
            continue
        lons.append(float(geom[0]))
        lats.append(float(geom[1]))
    return lons, lats


kept_lons, kept_lats = _coords(shared_anchors)
# Dropped entries are recorded in stats by id; recover their coords from raw_anchors.
dropped_ids = {d["id"] for d in shared_anchor_stats["dropped_anchors"]}
dropped_full = [a for a in raw_anchors if a.get("id") in dropped_ids]
drop_lons, drop_lats = _coords(dropped_full)

fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
plot_probes_on_map(
    ax,
    kept_lons,
    kept_lats,
    title=(
        f"Shared anchor eval corpus  kept={len(kept_lons)}  "
        f"(dropped {len(drop_lons)} same-ASN, marked ×)"
    ),
    label=f"kept anchor (n={len(kept_lons)})",
    color="#2ca02c",
    s=14,
    alpha=0.85,
)
ax.scer(
    drop_lons, drop_lats,
    s=180, c="#d62728", marker="x", linewidths=3,
    transform=ccrs.PlateCarree(), zorder=5,
    label=f"dropped (n={len(drop_lons)})",
)
ax.legend(loc="lower left", fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

NameError: name 'shared_anchors' is not defined

## Benchmark inputs: per-setup probe corpora

Loads the six per-setup probe JSONs from
[datasets/ripe_atlas/asn_corpora/](../../../datasets/ripe_atlas/asn_corpora/)
and shows the **filter funnel** for each setup —
`matched_asn → continent filter → city dedup → kept`. The two-step filter
makes it visible how much each guard contributes: the eyeball telcos lose
~25% of their probes to city-cell dedup while the cloud operators lose
~50% (multiple datacenter probes per metro collapsing to one). The maps
below plot each setup's final selected probes (red) over the full
sanitized corpus (faint gray).

In [18]:
# Same six setups as select_probes_and_anchors.py, in display order.
SETUPS = [
    (7922,  "Comcast",      "North America", "north_america"),
    (7018,  "",         "North America", "north_america"),
    (3209,  "Vodafone DE",  "Europe",        "europe"),
    (3215,  "Orange FR",    "Europe",        "europe"),
    (31898, "Oracle Cloud", "Global",        "global"),
    (16509, " AWS",   "Global",        "global"),
]

# Load each setup's probe JSON + stats JSON and reconstruct the filter funnel:
#   matched_asn  →  continent filter  →  city dedup  →  kept
# The continent filter has TWO guards: country_code → continent (catches
# e.g. cc=GB), and a coord bounding-box (catches cc=FR probes in Guadeloupe,
# cc=US probes in Hawaii, etc.).
per_setup_probes = {}
rows = []
for asn, operator, setup_continent, folder in SETUPS:
    probes_path = CORPORA_DIR / folder / f"probes_of_as_{asn}.json"
    stats_path = CORPORA_DIR / folder / f"probes_of_as_{asn}_stats.json"
    with open(probes_path) as fh:
        kept = json.load(fh)
    with open(stats_path) as fh:
        stats = json.load(fh)
    per_setup_probes[asn] = {
        "operator": operator,
        "setup_continent": setup_continent,
        "folder": folder,
        "kept": kept,
        "stats": stats,
    }
    top_countries = ", ".join(
        f"{country_name(cc)}({n})"
        for cc, n in list(stats["kept_by_country"].items())[:3]
    )
    pre_dedup = stats.get("pre_dedup_kept", stats["kept"])
    dedup_dropped = stats.get("dedup_dropped", 0)
    dropped_cc = stats.get("dropped_other_continent", 0)
    dropped_coords = stats.get("dropped_coords_outside_continent", 0)
    rows.append({
        "asn": asn,
        "operator": operator,
        "setup_continent": setup_continent,
        "matched_asn": stats["matched_asn"],
        "dropped_cc": dropped_cc,
        "dropped_coords": dropped_coords,
        "after_continent": pre_dedup,
        "dedup_dropped": dedup_dropped,
        "kept": stats["kept"],
        "n_cells": stats.get("n_city_cells", pre_dedup),
        "n_countries": len(stats["kept_by_country"]),
        "top_countries": top_countries,
    })

setup_probe_stats = pd.DataFrame(rows).set_index("asn")
print(f"Loaded {len(SETUPS)} probe corpora from {CORPORA_DIR}")
print(f"  full sanitized probe corpus: {len(all_df):,}")
print(f"  filter funnel totals across all setups:")
print(f"    matched_asn        sum = {setup_probe_stats['matched_asn'].sum():>4}")
print(f"    dropped_cc         sum = {setup_probe_stats['dropped_cc'].sum():>4}  (country_code → wrong continent)")
print(f"    dropped_coords     sum = {setup_probe_stats['dropped_coords'].sum():>4}  (cc passed but coords outside continent bbox)")
print(f"    after_continent    sum = {setup_probe_stats['after_continent'].sum():>4}")
print(f"    dedup_dropped      sum = {setup_probe_stats['dedup_dropped'].sum():>4}")
print(f"    kept (final)       sum = {setup_probe_stats['kept'].sum():>4}")
print()
setup_probe_stats

FileNotFoundError: [Errno 2] No such file or directory: '/home/nuwinslab/workspacecbg-framework/datasets/ripe_atlas/asn_corpora/north_america/probes_of_as_7922.json'

In [19]:
# 2x3 grid of per-setup probe maps. Selected probes in red, full sanitized
# corpus underplotted in faint gray for context.
n_cols = 3
n_rows = 2
fig = plt.figure(figsize=(22, 11))

bg_lons = all_df["lon"].to_numpy()
bg_lats = all_df["lat"].to_numpy()

for i, (asn, operator, setup_continent, folder) in enumerate(SETUPS):
    info = per_setup_probes[asn]
    lons = [
        float((e.get("geometry") or {}).get("coordinates", [None, None])[0])
        for e in info["kept"]
        if (e.get("geometry") or {}).get("coordinates")
    ]
    lats = [
        float((e.get("geometry") or {}).get("coordinates", [None, None])[1])
        for e in info["kept"]
        if (e.get("geometry") or {}).get("coordinates")
    ]
    n_countries = info["stats"]["kept_by_country"]
    cw = "country" if len(n_countries) == 1 else "countries"
    ax = fig.add_subplot(n_rows, n_cols, i + 1, projection=ccrs.PlateCarree())
    plot_probes_on_map(
        ax,
        lons,
        lats,
        title=(
            f"AS{asn} — {operator} ({setup_continent})  "
            f"n={info['stats']['kept']}, {len(n_countries)} {cw}"
        ),
        color="#d62728",
        s=14,
        alpha=0.9,
        gridlines=False,
        background_lons=bg_lons,
        background_lats=bg_lats,
        title_fontsize=10,
    )

fig.suptitle(
    f"Per-setup probe corpora — selected probes (red) over full sanitized corpus (n={len(all_df):,}, gray)",
    fontsize=14,
    y=0.995,
)
plt.tight_layout()
plt.show()

KeyError: 7922

<Figure size 2200x1100 with 0 Axes>